# CUBE.AI: Multi-Agent Literature Review for Radiology Research

This notebook implements an automated literature review system for the **CUBE.AI** graduation project. 

### Project Scope:
1. **Phase 1 (Completed):** Interactive and assisted medical image segmentation.
2. **Phase 2 (Extension):** Multi-agent LLM systems as research assistants for radiology.

This system uses **Microsoft AutoGen** to orchestrate specialized agents that search for academic papers and synthesize them into a professional report suitable for a thesis or grant application.

In [1]:
# 1. Install necessary libraries
!pip install -U autogen-agentchat autogen-ext google-generativeai

   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.0 MB 1.4 MB/s eta 0:00:05
   ---- ----------------------------------- 0.8/7.0 MB 1.5 MB/s eta 0:00:05
   ----- ---------------------------------- 1.0/7.0 MB 1.3 MB/s eta 0:00:05
   ------- -------------------------------- 1.3/7.0 MB 1.3 MB/s eta 0:00:05
   -------- ------------------------------- 1.6/7.0 MB 1.3 MB/s eta 0:00:05
   -------- ------------------------------- 1.6/7.0 MB 1.3 MB/s eta 0:00:05
   ---------- ----------------------------- 1.8/7.0 MB 1.3 MB/s eta 0:00:05
   ----------- ---------------------------- 2.1/7.0 MB 1.2 MB/s eta 0:00:04
   ------------- -------------------------- 2.4/7.0 MB 1.2 MB/s eta 0:00:05
   ------------- -------------------------- 2.4/7.0 MB 1.2 MB/s eta 0:00:05
   -------------- ------------------------- 2.6/7.0 MB 1.1 MB/s eta 0:00:05
   -------------- --------


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [4]:
import asyncio
import os
import urllib, urllib.request
import xml.etree.ElementTree as ET
from typing import Any, Mapping, Optional

# AutoGen Imports
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core.tools import FunctionTool

# Importing the custom client we created
from ollama_client import OllamaClient

ModuleNotFoundError: No module named 'ollama_client'

## Define Research Tools
We define a tool that allows agents to search Arxiv for the latest research in Medical AI and Agentic Systems.

In [ ]:
import urllib, urllib.request
import xml.etree.ElementTree as ET

def search_arxiv(query: str, max_results: int = 5) -> str:
    """Search Arxiv for academic papers based on a query."""
    base_url = 'http://export.arxiv.org/api/query?'
    params = urllib.parse.urlencode({'search_query': f'all:{query}', 'start': 0, 'max_results': max_results})
    
    with urllib.request.urlopen(base_url + params) as response:
        content = response.read().decode('utf-8')
    
    root = ET.fromstring(content)
    namespace = {'atom': 'http://www.w3.org/2005/Atom'}
    
    results = []
    for entry in root.findall('atom:entry', namespace):
        title = entry.find('atom:title', namespace).text.strip()
        summary = entry.find('atom:summary', namespace).text.strip()
        link = entry.find('atom:id', namespace).text.strip()
        results.append(f"TITLE: {title}\nLINK: {link}\nSUMMARY: {summary[:300]}...\n")
    
    return "\n---\n".join(results) if results else "No papers found."

arxiv_tool = FunctionTool(search_arxiv, description="Search Arxiv for medical and AI research papers.")

## Initialize Agents
We define three distinct roles to ensure high academic quality.

In [ ]:
# The Researcher: Finds the data
search_agent = AssistantAgent(
    name="Arxiv_Researcher",
    model_client=model_client,
    tools=[arxiv_tool],
    system_message="""You are a research expert. Search for papers on: 
    1. Interactive medical image segmentation (SAM, foundation models in radiology).
    2. Multi-agent LLM systems in clinical workflows.
    Provide summaries of the most relevant papers."""
)

# The Specialist: Evaluates technical feasibility and radiology relevance
radiology_agent = AssistantAgent(
    name="Radiology_Specialist",
    model_client=model_client,
    system_message="""You are a Radiology AI Specialist. Analyze the findings from the researcher.
    Focus on how these technologies help clinicians without AI expertise.
    Critique the methods and suggest how CUBE.AI can fill gaps in existing research."""
)

# The Writer: Compiles the final report
writer_agent = AssistantAgent(
    name="Thesis_Writer",
    model_client=model_client,
    system_message="""You are an academic writer. Synthesize the research into a formal literature review.
    Structure it with: 
    - Introduction to Assisted Segmentation.
    - Recent advances in Multi-Agent LLMs for Healthcare.
    - Future Directions for CUBE.AI.
    End your final response with the word 'TERMINATE'."""
)

termination = TextMentionTermination("TERMINATE")
team = RoundRobinGroupChat([search_agent, radiology_agent, writer_agent], termination_condition=termination)

## Execute the Review
This will run the multi-agent conversation and output the final synthesized literature review.

In [ ]:
task_description = """
Perform a literature review for the CUBE.AI project.
Focus on: 
1. Efficient 'human-in-the-loop' segmentation for CT/MRI using foundation models.
2. Multi-agent systems that act as research assistants or 'Co-pilots' for radiologists.
Highlight why providing these tools to clinics without AI experts is a novel research direction.
"""

await Console(team.run_stream(task=task_description))